# AgentSHAP — Runnable Notebook

This notebook shows how to use AgentSHAP to explain which tools an AI agent
relied on to answer a given prompt. It uses real tools from the API-Bank benchmark.

**Required files (in `token_shap/`):**
- `__init__.py` (simplified)
- `base.py`
- `agent_shap.py`
- `tools.py`

**Required packages:**
```
pip install openai numpy pandas matplotlib scikit-learn tqdm sentence-transformers
```

## 1. Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add the repo root to path so we can import token_shap
repo_root = Path().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from token_shap import AgentSHAP, create_function_tool
from token_shap.base import OpenAIModel, OpenAIEmbeddings, TfidfTextVectorizer

print('Imports OK')

In [ ]:
# Set your OpenAI API key here, or export it as an environment variable
# before launching the notebook:
#   export OPENAI_API_KEY=sk-...

API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY_HERE')

if API_KEY == 'YOUR_API_KEY_HERE':
    raise ValueError('Please set your OPENAI_API_KEY environment variable or paste it above.')

print('API key loaded.')

## 2. Define Tools

We create three simple tools that simulate real APIs:
- **Calculator** — evaluate arithmetic expressions
- **QueryStock** — look up a stock price
- **Wiki** — search Wikipedia

In [ ]:
# --- Calculator ---
def calculator_fn(args):
    expression = args.get('expression', '0')
    allowed = set('0123456789+-*/.() ')
    if not all(c in allowed for c in expression):
        return 'Error: Invalid expression'
    try:
        return str(eval(expression))
    except Exception as e:
        return f'Error: {e}'

calculator_tool = create_function_tool(
    name='Calculator',
    description='Evaluate arithmetic expressions. Use this for any math calculation.',
    parameters={
        'type': 'object',
        'properties': {
            'expression': {
                'type': 'string',
                'description': 'A math expression to evaluate, e.g. "(5+6)*3"'
            }
        },
        'required': ['expression']
    },
    executor=calculator_fn
)

# --- Stock Query ---
STOCK_DB = {
    'AAPL':  {'2024-01-15': 185.92, '2024-01-16': 183.63},
    'GOOGL': {'2024-01-15': 141.80, '2024-01-16': 140.23},
    'MSFT':  {'2024-01-15': 388.47, '2024-01-16': 390.11},
    'TSLA':  {'2024-01-15': 218.89, '2024-01-16': 212.19},
    'SQ':    {'2022-03-14': 92.34,  '2022-03-15': 89.12},
}

def query_stock_fn(args):
    symbol = args.get('symbol', '').upper()
    date   = args.get('date', '')
    if symbol in STOCK_DB:
        prices = STOCK_DB[symbol]
        if date in prices:
            return f'{symbol} on {date}: ${prices[date]}'
        # Return the most recent price if no date match
        latest_date = sorted(prices.keys())[-1]
        return f'{symbol} (latest, {latest_date}): ${prices[latest_date]}'
    return f'No data found for symbol {symbol}'

stock_tool = create_function_tool(
    name='QueryStock',
    description='Look up the stock price for a given ticker symbol and optional date.',
    parameters={
        'type': 'object',
        'properties': {
            'symbol': {
                'type': 'string',
                'description': 'Stock ticker symbol, e.g. "AAPL" or "TSLA"'
            },
            'date': {
                'type': 'string',
                'description': 'Date in YYYY-MM-DD format (optional)'
            }
        },
        'required': ['symbol']
    },
    executor=query_stock_fn
)

# --- Wikipedia Search ---
WIKI_DB = {
    'python':           'Python is a high-level, general-purpose programming language known for its readability.',
    'artificial intelligence': 'Artificial intelligence (AI) is the simulation of human intelligence in machines.',
    'machine learning': 'Machine learning is a subset of AI that enables systems to learn from data.',
    'newton':           'Isaac Newton was an English mathematician and physicist, famous for the laws of motion.',
    'einstein':         'Albert Einstein developed the theory of relativity and won the Nobel Prize in Physics in 1921.',
}

def wiki_fn(args):
    query = args.get('query', '').lower()
    for key, value in WIKI_DB.items():
        if key in query or query in key:
            return value
    return f'No Wikipedia article found for "{query}"'

wiki_tool = create_function_tool(
    name='Wiki',
    description='Search Wikipedia for factual information about a topic.',
    parameters={
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'The topic to search for on Wikipedia'
            }
        },
        'required': ['query']
    },
    executor=wiki_fn
)

tools = [calculator_tool, stock_tool, wiki_tool]
print(f'Tools defined: {[t.name for t in tools]}')

## 3. Initialise the Model and Vectorizer

We use `gpt-4o-mini` for the agent and OpenAI embeddings for measuring
response similarity. If you want a cheaper/offline option, swap the
vectorizer for `TfidfTextVectorizer()` — no API calls needed for similarity.

In [ ]:
model = OpenAIModel(model_name='gpt-4o-mini', api_key=API_KEY)

# Option A: OpenAI embeddings (semantic similarity, costs a tiny amount)
vectorizer = OpenAIEmbeddings(api_key=API_KEY, model='text-embedding-3-large')

# Option B: TF-IDF (free, offline, slightly less accurate)
# vectorizer = TfidfTextVectorizer()

print('Model and vectorizer ready.')

## 4. Run AgentSHAP — Single Prompt

We analyse a single prompt and print the Shapley values for each tool.

In [ ]:
agent_shap = AgentSHAP(
    model=model,
    tools=tools,
    vectorizer=vectorizer,
    max_iterations=3,
    debug=False          # Set to True to see detailed logs
)

prompt = 'What is the stock price of SQ on March 14th, 2022?'

results_df, shapley_values = agent_shap.analyze(
    prompt=prompt,
    sampling_ratio=0.5   # 0.0 = essential combinations only; 1.0 = exhaustive
)

print('\nShapley Values:')
for tool, value in sorted(shapley_values.items(), key=lambda x: x[1], reverse=True):
    print(f'  {tool:<15} {value:.4f}')

## 5. Visualise Results

In [ ]:
# Text-based importance ranking
agent_shap.print_tool_ranking()

In [ ]:
import matplotlib.pyplot as plt

# Bar chart
fig = agent_shap.plot_tool_importance(title=f'Tool Importance — "{prompt}"')
plt.show()

In [ ]:
# Coloured text (red = high importance, blue = low importance)
print('Coloured tool ranking:')
agent_shap.print_colored_tools()

In [ ]:
# Inspect the full results DataFrame
print('Combination results:')
results_df[['tools_available', 'similarity']].sort_values('similarity', ascending=False)

## 6. Compare Tool Importance Across Multiple Prompts

This shows how the same agent assigns different importance to tools
depending on what the user is asking.

In [ ]:
prompts = [
    'Can you calculate (5+6)*3 for me?',                          # Should highlight Calculator
    'What is the stock price of SQ on March 14th, 2022?',        # Should highlight QueryStock
    'Can you help me search artificial intelligence on Wikipedia?', # Should highlight Wiki
]

fig, all_shap_values = agent_shap.compare_prompts(
    prompts=prompts,
    sampling_ratio=0.5
)
plt.show()

## 7. Tool Usage Summary

After running `compare_prompts`, `agent_shap` stores the results from the
**last prompt** analysed. We can inspect the tool usage for that prompt.

In [ ]:
summary = agent_shap.get_tool_usage_summary()
print(f'Tool usage summary for last prompt ("{prompts[-1]}"):')
print(summary.to_string(index=False))

## 8. Detailed Results

You can access the full internal state after any `analyze()` call.

In [ ]:
details = agent_shap.get_detailed_results()

print('Prompt:', details['prompt'])
print()
print('Baseline response (all tools available):')
print(details['baseline_response'])
print()
print('Baseline tool usage:', details['baseline_tool_usage'])
print()
print('Shapley values:', details['shapley_values'])

## 9. Quick Reference

| Parameter | What it does |
|---|---|
| `sampling_ratio=0.0` | Only essential combinations (fast, less accurate) |
| `sampling_ratio=0.5` | Half of all possible combinations (balanced) |
| `sampling_ratio=1.0` | Exhaustive — all 2^n combinations (slow, most accurate) |
| `max_combinations=N` | Hard cap on total combinations regardless of ratio |
| `max_iterations=3` | Max tool-calling rounds per combination |
| `debug=True` | Print verbose logs during analysis |